# Experiment 1 -- aggregate and plot: error versus alignment (new pooled method)

Identical to `experiment1_01_aggregate_and_plot.ipynb`, except the "Pooled"
line uses `method="new_pooled"` (`target_source_pooled_subspace_estimate` --
pools the target's own estimated direction into the projection subspace
alongside the source's, rather than reserving it for a separate target-only
branch; not part of the discussion draft, a user-specified extension) instead
of `method="pooled"` (`pooled_subspace_estimate`, the original data-pooling
estimator from the discussion draft). This is done via a single filter/rename
right after loading the raw data (see the cell below the raw-CSV load) --
every other cell is untouched and keys off the literal string `"pooled"`
regardless of which estimator actually produced it.

**`new_pooled` was wired into `run_experiment1.py` but the Slurm array has
not been rerun** -- until it is, the raw CSVs on disk only have `method` in
{target, source, pooled, adaptive}, so this notebook's "Pooled" line will be
empty (the sanity-check cell below will show 0 seeds for `pooled` here,
since after the rename no rows carry that label yet).

Reads every per-task CSV written by `run_experiment1.py` into
`Results_simulation/experiment1/raw/`, aggregates the Monte Carlo mean and
standard error at each `(regime, mu, method)` grid point, writes the
combined table to `Results_simulation/experiment1/combined_new_pooled/`,
and produces the error-vs-alignment figure (one panel per regime, target
misclustering error against `mu`, one line per method).

In [1]:
import sys, os, glob

# Hardcoded (rather than relative to "..") because the kernel's cwd isn't
# guaranteed to be this notebook's directory -- e.g. VS Code's Jupyter
# extension often starts kernels from the workspace root instead. Mirrors
# the PROJECT_ROOT convention already used in Slurm_Scripts/*/run_*.sh.
PROJECT_ROOT = "/home/nandy.15/Research/Transfer_clustering"
sys.path.insert(0, os.path.join(PROJECT_ROOT, "Experiments_Script"))

# The figure below uses matplotlib's text.usetex=True, which shells out to
# `latex`/`dvipng`. Hardcoded (rather than relying on `module load texlive`
# having been run in the launching shell) because the Jupyter kernel's PATH
# is whatever VS Code/Jupyter started with -- it won't see a module loaded
# after the fact. Matches the cluster's `module show texlive/2025`.
TEXLIVE_BIN = "/opt/lmod/texlive/2025/bin/x86_64-linux"
if TEXLIVE_BIN not in os.environ["PATH"].split(os.pathsep):
    os.environ["PATH"] = TEXLIVE_BIN + os.pathsep + os.environ["PATH"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from run_experiment1 import REGIME_ORDER, MU_GRID, REGIMES

In [ ]:
RAW_DIR = os.path.join(PROJECT_ROOT, "Results_simulation", "experiment1", "raw")
COMBINED_DIR = os.path.join(PROJECT_ROOT, "Results_simulation", "experiment1", "combined_new_pooled")
os.makedirs(COMBINED_DIR, exist_ok=True)

paths = sorted(glob.glob(os.path.join(RAW_DIR, "*.csv")))
print(f"Found {len(paths)} raw result files")
assert paths, f"No CSVs found in {RAW_DIR} -- has the SLURM array finished any tasks yet?"

df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
df.head()

In [ ]:
# Swap in the new pooled method: drop the original "pooled" rows, then
# rename "new_pooled" -> "pooled" so every downstream cell (which keys off
# the literal string "pooled" -- METHOD_ORDER/COLORS/LABELS, groupby,
# plotting) is unchanged and now reflects target_source_pooled_subspace_estimate.
df = df[df["method"] != "pooled"].copy()
df.loc[df["method"] == "new_pooled", "method"] = "pooled"
df.head()

In [3]:
# Sanity check: how many distinct seeds actually landed per (regime, method)?
# If this is well short of N_SEEDS in run_experiment1.py, the array job
# hasn't finished (or some tasks failed -- check Slurm_Scripts/.../Error_Messages).
df.groupby(["regime", "method"])["seed"].nunique().unstack()

method,adaptive,pooled,source,target
regime,,,,
R1,100,100,100,100
R2,100,100,100,100
R3,100,100,100,100


In [4]:
summary = (
    df.groupby(["regime", "mu", "method"])["error"]
      .agg(mean_error="mean", se_error=lambda s: s.std(ddof=1) / np.sqrt(len(s)), n="count")
      .reset_index()
)
combined_path = os.path.join(COMBINED_DIR, "experiment1_summary.csv")
summary.to_csv(combined_path, index=False)
print(f"Wrote {combined_path}")
summary.head(12)

Wrote /home/nandy.15/Research/Transfer_clustering/Results_simulation/experiment1/combined/experiment1_summary.csv


,regime,mu,method,mean_error,se_error,n
0,R1,0.000,adaptive,0.47100,0.001931,100
1,R1,0.000,pooled,0.46540,0.002569,100
2,R1,0.000,source,0.47065,0.001950,100
3,R1,0.000,target,0.46595,0.002353,100
4,R1,0.025,adaptive,0.47155,0.002313,100
5,R1,0.025,pooled,0.47125,0.002122,100
6,R1,0.025,source,0.47330,0.002280,100
7,R1,0.025,target,0.46595,0.002353,100
8,R1,0.050,adaptive,0.47020,0.002249,100
9,R1,0.050,pooled,0.46460,0.002675,100


## Figure: error versus alignment

One panel per regime (R1, R2, R3), `mu` on the x-axis, Monte Carlo mean
target misclustering error on a shared y-axis (so magnitudes are directly
comparable across regimes), one line per method with SE error bars.
Colors are fixed per method across all three panels (never reassigned),
consistent with a small-multiples layout: blue = target-only (baseline),
orange = source-only, purple = **pooled** (here: `new_pooled` /
`target_source_pooled_subspace_estimate` -- pools the target's own
estimated direction into the projection subspace alongside the source's),
green = adaptive (the validation-statistic target/source switch).

In [ ]:
METHOD_COLORS = {"target": "#2a78d6", "source": "#eb6834", "pooled": "#8e44ad", "adaptive": "#008300"}
METHOD_LABELS = {"target": "Target-only", "source": "Source-only", "pooled": "Pooled", "adaptive": "Adaptive"}
METHOD_ORDER = ["target", "source", "pooled", "adaptive"]

# Smallest nonzero mu on the grid sets the linear-to-log crossover, so
# mu=0 still shows up (symlog can't take log of 0) while everything else
# is spaced by log2(mu).
LINTHRESH = min(mu for mu in MU_GRID if mu > 0)

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)

for ax, regime in zip(axes, REGIME_ORDER):
    sub = summary[summary["regime"] == regime]
    for method in METHOD_ORDER:
        m = sub[sub["method"] == method].sort_values("mu")
        ax.errorbar(
            m["mu"], m["mean_error"], yerr=m["se_error"],
            label=METHOD_LABELS[method], color=METHOD_COLORS[method],
            linewidth=2, marker="o", markersize=6, capsize=3,
        )
    cfg = REGIMES[regime]
    ax.set_title(rf"\texttt{{{regime}}} ($d={cfg['d']}$, $n_T={cfg['n_T']}$, $n_S={cfg['n_S']}$)")
    ax.set_xscale("symlog", base=2, linthresh=LINTHRESH)
    ax.set_xlabel(r"$\mu$ (log$_2$ scale)")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, alpha=0.25)

axes[0].set_ylabel("target misclustering error")
axes[0].legend(frameon=False, fontsize=9)
fig.suptitle("Experiment 1: error versus alignment ($\\Delta_T = 0.8$ fixed, new pooled method)", y=1.03)
fig.tight_layout()
fig.savefig(os.path.join(COMBINED_DIR, "experiment1_error_vs_alignment_new_pooled.pdf"), bbox_inches="tight")
plt.show()

## Diagnostic: calibrated C0 stability

`run_experiment1.py` also records `C0_used` for every `adaptive` row (the
bootstrap-calibrated constant from `calibrate_C0_bootstrap`, see
`Experiments_Script/transfer_clustering/two_community.py`). If it's roughly
stable within a regime (and across regimes, since it's meant to be a
universal constant), that supports caching a single calibrated `C0`
instead of re-bootstrapping per dataset.

In [6]:
c0 = df[df["method"] == "adaptive"].copy()
c0["C0_used"] = pd.to_numeric(c0["C0_used"], errors="coerce")
c0.groupby(["regime", "mu"])["C0_used"].agg(["mean", "std", "count"])

mean       std  count
regime mu                              
R1     0.000  1.209143  0.012394    100
       0.025  1.209143  0.012394    100
       0.050  1.209143  0.012394    100
       0.100  1.209143  0.012394    100
       0.150  1.209143  0.012394    100
       0.200  1.209143  0.012394    100
       0.400  1.209143  0.012394    100
       0.800  1.209143  0.012394    100
R2     0.000  1.209143  0.012394    100
       0.025  1.209143  0.012394    100
       0.050  1.209143  0.012394    100
       0.100  1.209143  0.012394    100
       0.150  1.209143  0.012394    100
       0.200  1.209143  0.012394    100
       0.400  1.209143  0.012394    100
       0.800  1.209143  0.012394    100
R3     0.000  1.029651  0.002093    100
       0.025  1.029651  0.002093    100
       0.050  1.029651  0.002093    100
       0.100  1.029651  0.002093    100
       0.150  1.029651  0.002093    100
       0.200  1.029651  0.002093    100
       0.400  1.029651  0.002093    100
       0.800  1.029651  0.002093    100